In [1]:
! pip3 install --upgrade --quiet --user google-cloud-aiplatform==1.88.0

In [2]:
# Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [2]:
import vertexai

PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

print(PROJECT_ID)

vertexai.init(project=PROJECT_ID, location=LOCATION)

qwiklabs-gcp-03-341e958fd7a4


In [3]:
import requests
from vertexai.generative_models import (
    Content,
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Part,
    Tool,
)

In [4]:
def add_func(a: float, b: float) -> float:
    print("Calling add function")
    return a + b

def multiply_func(a: float, b: float) -> float:
    print("Calling multiply function")
    return a * b

In [5]:
# Test manual function
print(add_func(5,9))
print(multiply_func(12,3))

Calling add function
14
Calling multiply function
36


In [30]:
add_func_declaration = FunctionDeclaration(
    name="add_function",
    description="add two numerical inputs and return the result",
    parameters={
        "type": "OBJECT",
        "properties": {
            "a": {"type": "NUMBER", "description": "first numerical input"},
            "b": {"type": "NUMBER", "description": "second numerical input"}
        },
    },
)

multiply_func_declaration = FunctionDeclaration(
    name="multiply_function",
    description="multiply two numerical inputs and return the result",
    parameters={
        "type": "OBJECT",
        "properties": {
            "a": {"type": "NUMBER", "description": "first numerical input"},
            "b": {"type": "NUMBER", "description": "second numerical input"}
        },
    },
)

In [31]:
math_tool = Tool(
    function_declarations=[add_func_declaration, multiply_func_declaration]
)

In [32]:
model = GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[math_tool],
    generation_config=GenerationConfig(temperature=0),
    system_instruction=[
        "Fulfill the user's instructions, including telling jokes.",
        "Answer the user's question using the appropriate function tool if available.",
        "You may call both the 'multiply' and 'add' functions one after the other if needed.",
        "Use the 'multiply' function only for multiplication-related tasks.",
        "Use the 'add' function only for addition-related tasks.",
        "A function tool may only be invoked if its capabilities are an exact match for the described task. Avoid speculative or improper use of tools.",
        "In the absence of a suitable tool, respond to the user with a generic answer without performing the mathematical calculation yourself.",
    ]
)

In [33]:
chat = model.start_chat()

In [52]:
def handle_response(response):

    # If there is a function call then invoke it
    # Otherwise print the response.
    if response.candidates[0].function_calls:
        function_call = response.candidates[0].function_calls[0]
    else:
        print(response.text)
        return

    # TODO: Complete the following sections
    if function_call.name == "add_function":
        # Extract the arguments to use in your function
        a = function_call.args["a"]
        b = function_call.args["b"]
        # Call your function
        result = add_func(a, b)
        # Send the result back to the chat session with the model
        response = chat.send_message(
            Part.from_function_response(
                name="add",
                response={
                    "content": result,
                },
            ),
        )
        # Make a recursive call of this handler function
        handle_response(response)
        

    elif function_call.name == "multiply_function":
        # Extract the arguments to use in your function
        a = function_call.args["a"]
        b = function_call.args["b"]
        # Call your function
        result = multiply_func(a, b)
        # Send the result back to the chat session with the model
        response = chat.send_message(
            Part.from_function_response(
                name="multiply",
                response={
                    "content": result,
                },
            ),
        )
        # Make a recursive call of this handler function
        handle_response(response)
        
    else:
        # You shouldn't end up here
        print(function_call)

In [72]:
response = chat.send_message("Tell me a joke?")
handle_response(response)

Why don't scientists trust atoms?

Because they make up everything!


In [ ]:
response = chat.send_message("I have 7 pizzas each with 16 slices. How many slices do I have?")
handle_response(response)

In [74]:
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. How many pizzas did they bring together?")
handle_response(response)

They brought 7 pizzas together.


In [ ]:
response = chat.send_message("Doug brought 3 pizzas. Andrew brought 4 pizzas. There are 16 slices per pizza. How many slices are there?")
handle_response(response)

In [ ]:
response = chat.send_message("Doug brought 4 pizzas, but Andrew dropped 2 on the ground. How many pizzas are left?")
handle_response(response)